In [ ]:
# Préparation commune des TP Python
# Le notebook utilise uniquement les ressources fournies dans le dossier codes/.
from pathlib import Path
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des paquets manquants :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

WORKDIR = Path.cwd().resolve()
CODES_DIR = WORKDIR.parent if WORKDIR.name == "correction" else WORKDIR
TOOLBOX_DIR = CODES_DIR / "toolbox"
DATA_DIR = CODES_DIR / "data"
if not CODES_DIR.is_dir():
    raise FileNotFoundError("Exécutez le notebook depuis le dossier codes/." )
if TOOLBOX_DIR.is_dir():
    sys.path.insert(0, str(TOOLBOX_DIR))

DICE_FILE = TOOLBOX_DIR / "DICE.py"
if not DICE_FILE.is_file():
    raise FileNotFoundError(
        f"Module du cours introuvable : {DICE_FILE}. "
        "Téléchargez le dossier codes complet, avec toolbox/."
    )

DATA_FILE = DATA_DIR / "ngfs_scenarios.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Donnée du cours introuvable : {DATA_FILE}. "
        "Téléchargez le dossier codes complet, avec data/."
    )

print("Environnement prêt :", CODES_DIR)


# TP4 — Scénarios de transition et prix du carbone

**Date de la séance :** mercredi 9 septembre 2026, 10:45–12:45

**Objectifs**
- Simuler des politiques climatiques contrastées avec DICE (variable de contrôle `mu`)
- Comparer transition précoce et transition tardive
- Manipuler les scénarios NGFS (prix du carbone, émissions, température)

**Prérequis** : TP2, TP3.

## Partie 1 — Scénarios de transition

**Question 1.** Simulez trois scénarios de politique climatique :
- Aucune action : mu=0.03 (baseline)
- Transition modérée : mu=0.3 dès 2025
- Transition forte : mu=0.7 dès 2025

In [ ]:
import DICE, numpy as np, pandas as pd, matplotlib.pyplot as plt

p_base = DICE.Params()
nT = p_base.nT

scenarios_mu = {
    'Aucune action (mu=0.03)': 0.03,
    'Transition modérée (mu=0.3)': 0.3,
    'Transition forte (mu=0.7)': 0.7
}
paths_transition = {}
for name, mu_val in scenarios_mu.items():
    p = DICE.Params()
    path = DICE.init_states(p)
    path[1:, p.i_mu] = mu_val
    timevec = range(1, p.nT)
    path = DICE.update_path(path, timevec, p)
    paths_transition[name] = (p, path)

In [ ]:
# Expected output: les émissions et la température baissent avec l'intensité de l'abattement mu
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, (p, path) in paths_transition.items():
    axes[0].plot(path[:, p.i_time], path[:, p.i_E], label=name, linewidth=2)
    axes[1].plot(path[:, p.i_time], path[:, p.i_T_AT], label=name, linewidth=2)
axes[0].set_title('Émissions E'); axes[0].set_xlabel('Année'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Température T_AT'); axes[1].set_xlabel('Année'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

**Question 2.** Calculez la réduction d'émissions cumulées entre 2025 et 2100 pour chaque scénario par rapport au scénario de référence.

In [ ]:
# Expected output: la réduction cumulée croît avec mu ; la transition forte réduit fortement les émissions cumulées
ref_name = 'Aucune action (mu=0.03)'
p_ref, path_ref = paths_transition[ref_name]
mask_2025 = path_ref[:, p_ref.i_time] >= 2025
E_ref_cum = np.nansum(path_ref[mask_2025, p_ref.i_E])

for name, (p, path) in paths_transition.items():
    mask = path[:, p.i_time] >= 2025
    E_cum = np.nansum(path[mask, p.i_E])
    reduction = E_ref_cum - E_cum
    print(f"{name}: émissions cumulées 2025-2100 = {E_cum:.0f} GtCO2, réduction vs référence = {reduction:.0f} GtCO2")

## Partie 2 — Transition ordonnée vs désordonnée

**Question 3.** Comparez deux politiques :
- Action précoce : mu=0.5 dès 2025
- Action tardive : mu=0.5 seulement à partir de 2040

In [ ]:
# Expected output: l'action tardive conduit à une température plus élevée en 2100 malgré le même niveau final de mu
p_early = DICE.Params()
path_early = DICE.init_states(p_early)
path_early[1:, p_early.i_mu] = 0.5
path_early = DICE.update_path(path_early, range(1, p_early.nT), p_early)

p_late = DICE.Params()
path_late = DICE.init_states(p_late)
year_col = p_late.t0 + p_late.Delta * (np.arange(1, p_late.nT))
idx_2040 = np.where(year_col >= 2040)[0][0] + 1
path_late[1:idx_2040, p_late.i_mu] = 0.03
path_late[idx_2040:, p_late.i_mu] = 0.5
path_late = DICE.update_path(path_late, range(1, p_late.nT), p_late)

plt.figure(figsize=(9, 5))
plt.plot(path_early[:, p_early.i_time], path_early[:, p_early.i_T_AT], label='Action précoce (dès 2025)', linewidth=2)
plt.plot(path_late[:, p_late.i_time], path_late[:, p_late.i_T_AT], label='Action tardive (dès 2040)', linewidth=2, linestyle='--')
plt.xlabel('Année'); plt.ylabel('T_AT (°C)')
plt.title('Transition ordonnée vs transition retardée')
plt.legend(); plt.grid(True)
plt.show()

print(f"T_AT 2100 (action précoce) : {path_early[-1, p_early.i_T_AT]:.2f}°C")
print(f"T_AT 2100 (action tardive) : {path_late[-1, p_late.i_T_AT]:.2f}°C")

## Partie 3 — Scénarios NGFS

**Question 4.** Chargez les données NGFS et tracez les trajectoires de prix du carbone et d'émissions pour les trois scénarios.

In [ ]:
# Expected output: NetZero2050 a le prix du carbone le plus élevé et les émissions les plus basses en fin de période ;
# CurrentPolicies n'a pas de prix du carbone et des émissions quasi stables
df_ngfs = pd.read_csv(DATA_DIR / "ngfs_scenarios.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, grp in df_ngfs.groupby('scenario'):
    axes[0].plot(grp['year'], grp['carbon_price_usd'], label=name, linewidth=2)
    axes[1].plot(grp['year'], grp['emissions_gtco2'], label=name, linewidth=2)
axes[0].set_title('Prix du carbone (USD/tCO2)'); axes[0].set_xlabel('Année'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Émissions (GtCO2/an)'); axes[1].set_xlabel('Année'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

## Interprétation

### Éléments de réponse

- Plus l'abattement `mu` est élevé et précoce, plus les émissions cumulées et la température finale sont réduites : il existe un arbitrage direct entre effort de transition et réchauffement évité.
- Retarder l'action climatique au même niveau d'ambition finale (`mu=0.5`) conduit à une température plus élevée en 2100 : le carbone déjà émis reste dans l'atmosphère pendant des siècles, donc chaque année de retard a un coût cumulatif irréversible sur la trajectoire de température.
- Les scénarios NGFS structurent ce même arbitrage en 4 familles utilisées par les superviseurs financiers : une transition ordonnée et précoce limite le risque physique mais implique un prix du carbone élevé (risque de transition) ; une transition désordonnée ou retardée combine les deux risques.